In [ ]:
# %% [code]
%load_ext autoreload
%autoreload 2

#####################
# Standard Library  #
#####################
import os
import copy
import json

#####################
# Third-party       #
#####################
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Subset, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm, trange

#####################
# QONNX & FINN      #
#####################
# QONNX core and transformations
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.datatype import DataType
from qonnx.transformation.batchnorm_to_affine import BatchNormToAffine
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import (
    ConvertSubToAdd,
    ConvertDivToMul,
    GiveReadableTensorNames,
    GiveUniqueNodeNames,
    SortGraph,
    RemoveUnusedTensors,
    GiveUniqueParameterTensors,
    RemoveStaticGraphInputs,
    ApplyConfig,
)
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.insert_topk import InsertTopK
from qonnx.transformation.make_input_chanlast import MakeInputChannelsLast
from qonnx.transformation.double_to_single_float import DoubleToSingleFloat
from qonnx.transformation.remove import RemoveIdentityOps
from qonnx.custom_op.registry import getCustomOp
from qonnx.util.basic import (
    calculate_matvec_accumulator_range,
    interleave_matrix_outer_dim_from_partitions,
    roundup_to_integer_multiple,
)
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.util.config import extract_model_config_to_json

# FINN streamlining & dataflow transformations
import finn.transformation.fpgadataflow.convert_to_hls_layers as to_hls
from finn.transformation.streamline.absorb import (
    AbsorbScalarMulAddIntoTopK,
    AbsorbAddIntoMultiThreshold,
    AbsorbMulIntoMultiThreshold,
    FactorOutMulSignMagnitude,
    Absorb1BitMulIntoMatMul,
    Absorb1BitMulIntoConv,
    AbsorbConsecutiveTransposes,
    AbsorbTransposeIntoMultiThreshold,
    AbsorbSignBiasIntoMultiThreshold,
)
from finn.transformation.streamline.collapse_repeated import (
    CollapseRepeatedAdd,
    CollapseRepeatedMul,
)
from finn.transformation.streamline.reorder import (
    MoveAddPastMul,
    MoveScalarMulPastMatMul,
    MoveScalarAddPastMatMul,
    MoveAddPastConv,
    MoveScalarMulPastConv,
    MoveScalarLinearPastInvariants,
    MoveMaxPoolPastMultiThreshold,
    MakeScaleResizeNHWC,
    MoveMulPastMaxPool,
    MoveLinearPastEltwiseAdd,
    MoveLinearPastFork,
    MakeMaxPoolNHWC,
)
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from finn.transformation.streamline.sign_to_thres import ConvertSignToThres

from finn.builder.build_dataflow_config import DataflowBuildConfig, ShellFlowType
from finn.transformation.fpgadataflow.prepare_ip import PrepareIP
from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP
from finn.transformation.fpgadataflow.replace_verilog_relpaths import ReplaceVerilogRelPaths
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.fpgadataflow.set_fifo_depths import (
    InsertAndSetFIFODepths,
    RemoveShallowFIFOs,
    SplitLargeFIFOs,
)
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.insert_fifo import InsertFIFO
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild
from finn.custom_op.fpgadataflow.matrixvectoractivation import MatrixVectorActivation as MVAU
from finn.custom_op.fpgadataflow.thresholding_batch import Thresholding_Batch
from finn.util.visualization import showSrc, showInNetron
from finn.util.test import get_test_model_trained
from finn.util.data_packing import (
    npy_to_rtlsim_input,
    numpy_to_hls_code,
    pack_innermost_dim_as_hex_string,
    rtlsim_output_to_npy,
)
from finn.util.basic import pynq_part_map

#####################
# Brevitas          #
#####################
import brevitas
import brevitas.nn as qnn
from brevitas.export import export_qonnx
from brevitas.core.quant import QuantType
from brevitas.quant import (
    Int8Bias,
    Int8ActPerTensorFloat, 
    SignedBinaryWeightPerTensorConst, 
    SignedTernaryWeightPerTensorConst,
    SignedBinaryActPerTensorConst
)

#####################
# Visualization     #
#####################
import netron

#####################
# Device Setup      #
#####################
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Target device:", device)
print("Torch version:", torch.__version__)
print("Brevitas version:", brevitas.__version__)


In [ ]:
set_weight_bit_width = 4
set_activation_bit_width = 4
d = 5

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        self.double_conv = nn.Sequential(
            
            qnn.QuantConv2d(in_channels, out_channels, kernel_size = 3, weight_bit_width=set_weight_bit_width,
                            padding = 1, bias=False, return_quant_tensor=True),
            #qnn.QuantConv2d(in_channels, out_channels, kernel_size = 3, weight_quant=SignedBinaryWeightPerTensorConst,
            #                padding = 1, bias=False, return_quant_tensor=True),
            nn.BatchNorm2d(out_channels),
            qnn.QuantReLU(bit_width=set_activation_bit_width, return_quant_tensor=True),
            
            qnn.QuantConv2d(out_channels, out_channels, kernel_size = 3, weight_bit_width=set_weight_bit_width,
                            padding = 1, bias=False, return_quant_tensor=True),
            #qnn.QuantConv2d(out_channels, out_channels, kernel_size = 3, weight_quant=SignedBinaryWeightPerTensorConst,
            #                padding = 1, bias=False, return_quant_tensor=True),
            nn.BatchNorm2d(out_channels),
            qnn.QuantReLU(bit_width=set_activation_bit_width, return_quant_tensor=True)
        )

    def forward(self, x):
        return self.double_conv(x)



class Encoder(nn.Module):
    def __init__(self, scale, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            
            nn.MaxPool2d(scale), 
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)



class Upsampling(nn.Module):
    def __init__(self, scale, in_channels, out_channels):
        super().__init__()
        
        self.up = nn.Sequential(
            
            qnn.QuantConv2d(in_channels, out_channels, kernel_size=1, weight_bit_width=set_weight_bit_width, 
                            bias=False, return_quant_tensor=True),
            #qnn.QuantConv2d(in_channels, in_channels//2, kernel_size=1, weight_quant=SignedBinaryWeightPerTensorConst, 
            #                bias=False, return_quant_tensor=True),
            nn.BatchNorm2d(out_channels),
            qnn.QuantReLU(bit_width=set_activation_bit_width, return_quant_tensor=True),
            qnn.QuantUpsamplingNearest2d(scale_factor=scale, return_quant_tensor=True)
        )
        self.quant_inp = qnn.QuantIdentity(act_quant = Int8ActPerTensorFloat, bit_width=set_activation_bit_width, return_quant_tensor=True)

    def forward(self, x1, x2):
        # x1 is the feature map after upsampling
        # x2 is the feature map from the corresponding layer in the downsampling path for concatenation
        x1 = self.up(x1)
        x1 = self.quant_inp(x1)
        x2 = self.quant_inp(x2)
        x = x1 + x2
        x = self.quant_inp(x)
        
        return x



class OutConv(nn.Module):
    def __init__(self, d, in_channels, out_channels):
        super().__init__()
        self.conv = qnn.QuantConv2d(in_channels, out_channels, kernel_size = 1, weight_bit_width=set_weight_bit_width, bias=False)
        self.flatten = nn.Flatten()
        self.linear = qnn.QuantLinear(out_channels*(d+1)*(d+1), 1, weight_bit_width=set_weight_bit_width, bias=False)
        #self.conv = qnn.QuantConv2d(in_channels, out_channels, kernel_size = 1, weight_quant=SignedBinaryWeightPerTensorConst, bias=False)

    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        x = self.linear(x)

        return x
        # return self.conv(x)


In [ ]:
class EA_LAYER_1(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.quant_inp = qnn.QuantIdentity(bit_width=set_activation_bit_width, return_quant_tensor=True)
        self.double_conv1 = DoubleConv(in_channels, out_channels)
        self.double_conv5 = DoubleConv(out_channels, 8)
        self.output_conv6 = OutConv(8, 4)
    
    def forward(self, x):
        x = self.quant_inp(x)
        x1 = self.double_conv1(x)
        x = self.double_conv5(x1)
        x = self.output_conv6(x)

        # return x, x1
        return x
        

class EA_LAYER_2(nn.Module):
    def __init__(self, previous_model, in_channels, out_channels):
        super().__init__()

        self.quant_inp = previous_model.quant_inp
        self.double_conv1 = previous_model.double_conv1
        self.double_conv5 = previous_model.double_conv5
        self.output_conv6 = previous_model.output_conv6

        self.encode2 = Encoder(2, in_channels, out_channels) # (16, 32)
        self.double_conv4_1 = DoubleConv(out_channels, in_channels)
        self.upsample4_2 = Upsampling(2, in_channels, in_channels)
    
    def forward(self, x):
        x = self.quant_inp(x)
        x1 = self.double_conv1(x)
        x = self.encode2(x1)
        x = self.double_conv4_1(x)
        x = self.upsample4_2(x, x1)
        x = self.double_conv5(x)
        x = self.output_conv6(x)

        # return x, x1
        return x
    

class EA_LAYER_3(nn.Module):
    def __init__(self, previous_model, in_channels, out_channels):
        super().__init__()

        self.quant_inp = previous_model.quant_inp
        self.double_conv1 = previous_model.double_conv1
        self.encode2 = previous_model.encode2
        self.double_conv4_1 = previous_model.double_conv4_1
        self.upsample4_2 = previous_model.upsample4_2
        self.double_conv5 = previous_model.double_conv5
        self.output_conv6 = previous_model.output_conv6

        self.encode3 = Encoder(3, in_channels, out_channels) # (32, 64)
        self.upsample3 = Upsampling(3, out_channels, in_channels)
    
    def forward(self, x):
        x = self.quant_inp(x)
        x1 = self.double_conv1(x)
        x2 = self.encode2(x1)
        x = self.encode3(x2)
        x = self.upsample3(x, x2)
        x = self.double_conv4_1(x)
        x = self.upsample4_2(x, x1)
        x = self.double_conv5(x)
        x = self.output_conv6(x)

        # return x, x1
        return x

class Unet_circuit_2L(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.quant_inp = qnn.QuantIdentity(bit_width=set_activation_bit_width, return_quant_tensor=True)
        self.double_conv1 = DoubleConv(6, 16)
        self.encode2 = Encoder(2, 16, 32) # (16, 32)
        self.upsample4_2 = Upsampling(2, 32, 16)
        self.double_conv5 = DoubleConv(16, 8)
        self.output_conv6 = OutConv(d, 8, 4)
    
    
    def forward(self, x):
        x = self.quant_inp(x)
        x1 = self.double_conv1(x)
        x = self.encode2(x1)
        x = self.upsample4_2(x, x1)
        x = self.double_conv5(x)
        x = self.output_conv6(x)

        return x
    
class Unet_circuit_3L(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.quant_inp = qnn.QuantIdentity(bit_width=set_activation_bit_width, return_quant_tensor=True)
        self.double_conv1 = DoubleConv(10, 16)
        self.encode2 = Encoder(2, 16, 32) # (16, 32)
        self.encode3 = Encoder(3, 32, 64)
        self.upsample3 = Upsampling(3, 64, 32)
        self.double_conv4_1 = DoubleConv(32, 16)
        self.upsample4_2 = Upsampling(2, 16, 16)
        self.double_conv5 = DoubleConv(16, 8)
        self.output_conv6 = OutConv(d, 8, 4)
    
    
    def forward(self, x):
        x = self.quant_inp(x)
        x1 = self.double_conv1(x)
        x2 = self.encode2(x1)
        x = self.encode3(x2)
        x = self.upsample3(x, x2)
        x = self.double_conv4_1(x)
        x = self.upsample4_2(x, x1)
        x = self.double_conv5(x)
        x = self.output_conv6(x)

        return x
    
class Unet_circuit_4L(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.quant_inp = qnn.QuantIdentity(bit_width=set_activation_bit_width, return_quant_tensor=True)
        self.double_conv1 = DoubleConv(14, 16)
        self.encode2 = Encoder(2, 16, 32) # (16, 32)
        self.encode3 = Encoder(2, 32, 64)
        self.encode4 = Encoder(2, 64, 128)
        self.upsample4 = Upsampling(2, 128, 64)
        self.double_conv5_1 = DoubleConv(64, 32)
        self.upsample5_2 = Upsampling(2, 32, 32)
        self.double_conv6_1 = DoubleConv(32, 16)
        self.upsample6_2 = Upsampling(2, 16, 16)
        self.double_conv7 = DoubleConv(16, 8)
        self.output_conv8 = OutConv(d, 8, 4)
    
    
    def forward(self, x):
        x = self.quant_inp(x)
        x1 = self.double_conv1(x)
        x2 = self.encode2(x1)
        x3 = self.encode3(x2)
        x = self.encode4(x3)
        x = self.upsample4(x, x3)
        x = self.double_conv5_1(x)
        x = self.upsample5_2(x, x2)
        x = self.double_conv6_1(x)
        x = self.upsample6_2(x, x1)
        x = self.double_conv7(x)
        x = self.output_conv8(x)

        return x

In [ ]:
def step_resnet50_streamline_linear(model: ModelWrapper):
    streamline_transformations = [
        MoveLinearPastFork(),
        MoveMulPastMaxPool(),
        AbsorbSignBiasIntoMultiThreshold(), # Absorb add into MultiThreshold
        AbsorbScalarMulAddIntoTopK(),  # before MoveAddPastMul to avoid int->float
        ConvertSubToAdd(),
        ConvertDivToMul(),
        RemoveIdentityOps(),
        CollapseRepeatedMul(),
        BatchNormToAffine(), # BatchNorm to mul + add (B <1, C, 1, 1>)
        ConvertSignToThres(),
        MoveAddPastMul(),
        MoveScalarAddPastMatMul(),
        MoveAddPastConv(),
        MoveScalarMulPastMatMul(),
        MoveScalarMulPastConv(),
        MoveScalarLinearPastInvariants(),
        MoveAddPastMul(),
        CollapseRepeatedAdd(),
        CollapseRepeatedMul(),
        AbsorbAddIntoMultiThreshold(),
        FactorOutMulSignMagnitude(),
        MoveMaxPoolPastMultiThreshold(),
        AbsorbMulIntoMultiThreshold(),
        Absorb1BitMulIntoMatMul(),
        Absorb1BitMulIntoConv(),
        RoundAndClipThresholds(),
    ]
    for trn in streamline_transformations:
        model = model.transform(trn)
        model = model.transform(GiveUniqueNodeNames())
    return model


def step_resnet50_streamline_nonlinear(model: ModelWrapper):
    streamline_transformations = [
        MoveLinearPastEltwiseAdd(),
        MoveLinearPastFork(),
    ]
    for trn in streamline_transformations:
        model = model.transform(trn)
        model = model.transform(GiveUniqueNodeNames())
    return model

def step_resnet50_streamline(model: ModelWrapper):

    for iter_id in range(4):
        model = step_resnet50_streamline_linear(model)
        model = step_resnet50_streamline_nonlinear(model)

        # big loop tidy up
        model = model.transform(RemoveUnusedTensors())
        model = model.transform(GiveReadableTensorNames())
        model = model.transform(InferDataTypes())
        model = model.transform(SortGraph())

    model = model.transform(DoubleToSingleFloat())

    return model

def step_resnet50_convert_to_hls(model: ModelWrapper):
    model.set_tensor_datatype(model.graph.input[0].name, DataType["UINT4"])
    model = model.transform(InferDataLayouts())

    model = model.transform(DoubleToSingleFloat())
    model = model.transform(InferDataTypes())
    model = model.transform(SortGraph())

    to_hls_transformations = [
        to_hls.InferAddStreamsLayer,
        LowerConvsToMatMul,
        to_hls.InferChannelwiseLinearLayer,
        MakeMaxPoolNHWC,
        to_hls.InferPool_Batch,
        AbsorbTransposeIntoMultiThreshold,
        RoundAndClipThresholds,
        to_hls.InferQuantizedMatrixVectorActivation,
        to_hls.InferThresholdingLayer,
        AbsorbConsecutiveTransposes,
        to_hls.InferConcatLayer,
        MakeScaleResizeNHWC,
        to_hls.InferUpsample,
        to_hls.InferConvInpGen,
        to_hls.InferDuplicateStreamsLayer,
        to_hls.InferLabelSelectLayer,
        #InferLabelSelectLayer_reshape,
        #MoveTransposePastTopK,
        AbsorbConsecutiveTransposes,
        InferDataLayouts
    ]
    for trn in to_hls_transformations:
        model = model.transform(trn())
        model = model.transform(InferDataLayouts())
        model = model.transform(GiveUniqueNodeNames())
        model = model.transform(InferDataTypes())

    model = model.transform(RemoveCNVtoFCFlatten())
    model = model.transform(GiveReadableTensorNames())
    model = model.transform(RemoveUnusedTensors())
    model = model.transform(SortGraph())

    return model

In [ ]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/HLS_unet/model_onnx_save/3L"
model_save = os.environ['FINN_ROOT'] + "/notebooks/HLS_unet/model_onnx_save"
model_save_path = model_save + "/D5_Noise0.006_Train.pth"
U_net_model = Unet_circuit_3L(d)
print("d = ", d)
U_net_model.load_state_dict(torch.load(model_save_path, map_location=device))

In [ ]:
U_net_model.cpu()
export_onnx_path = model_dir + "/unet_export.onnx"
export_qonnx(U_net_model, torch.randn(1, d*2, d+1, d+1), export_onnx_path)
qonnx_cleanup(export_onnx_path, out_file=export_onnx_path)
model = ModelWrapper(export_onnx_path)

model = model.transform(ConvertQONNXtoFINN())
model = model.transform(GiveUniqueParameterTensors())
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(RemoveStaticGraphInputs())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(InsertTopK(k=1, axis=1))
model = model.transform(InferShapes())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(InferDataLayouts())
model.save(model_dir + "/unet.onnx")
model = ModelWrapper(model_dir + "/unet.onnx")
model = step_resnet50_streamline(model)
model.save(model_dir + "/unet_streamline.onnx")
model = ModelWrapper(model_dir + "/unet_streamline.onnx")
model = step_resnet50_convert_to_hls(model)
model.save(model_dir + "/unet_hls.onnx")
parent_model = model.transform(CreateDataflowPartition())
parent_model.save(model_dir + "/unet_dataflow_parent.onnx")
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")
# save the dataflow partition with a different name for easier access
dataflow_model = ModelWrapper(dataflow_model_filename)
dataflow_model.save(model_dir + "/unet_dataflow_model.onnx")



In [ ]:
# Generate MVAU params.h to save weights

# set up loaded model and save_path
model = ModelWrapper(model_dir + "/unet_dataflow_model_copy.onnx")
save_path = os.path.join(model_dir, "MVAU_params_for_dataflow")
save_path_dat = os.path.join(model_dir, "MVAU_memstream_dat")
save_path_npy = os.path.join(model_dir, "MVAU_weights_npy")
# Check if the directory exists, if not, create it
if not os.path.exists(save_path):
    os.makedirs(save_path)
if not os.path.exists(save_path_npy):
    os.makedirs(save_path_npy)
if not os.path.exists(save_path_dat):
    os.makedirs(save_path_dat)
fc_layers = model.get_nodes_by_op_type("MatrixVectorActivation")

# for file generation
# max_SIMD = 64
# max_PE = 32
# for synthesis
max_SIMD = 64
max_PE = 32
# input_channels = 10
input_channels = 2
channel_1L = 16
channel_2L = 32
channel_3L = 64

# For FINN synthesis, is_padding should always be set to False
# True is only used for file generation for the custom design
weights_shape = [
    (input_channels, min(channel_1L, max_PE), 9 * max(1, int(channel_1L/max_PE)), True),
    (min(channel_1L, max_SIMD), min(channel_1L, max_PE), 9 * max(1, int(channel_1L/max_SIMD))*max(1, int(channel_1L/max_PE)), True),
    
    (min(channel_1L, max_SIMD), min(channel_2L, max_PE), 9 * max(1, int(channel_1L/max_SIMD))*max(1, int(channel_2L/max_PE)), False),
    (min(channel_2L, max_SIMD), min(channel_2L, max_PE), 9 * max(1, int(channel_2L/max_SIMD))*max(1, int(channel_2L/max_PE)), False),
    
    (min(channel_2L, max_SIMD), min(channel_3L, max_PE), 9 * max(1, int(channel_2L/max_SIMD))*max(1, int(channel_3L/max_PE)), False),
    (min(channel_3L, max_SIMD), min(channel_3L, max_PE), 9 * max(1, int(channel_3L/max_SIMD))*max(1, int(channel_3L/max_PE)), False),
    
    (min(channel_3L, max_SIMD), min(channel_2L, max_PE), 1 * max(1, int(channel_3L/max_SIMD))*max(1, int(channel_2L/max_PE)), False),
    
    (min(channel_2L, max_SIMD), min(channel_1L, max_PE), 9 * max(1, int(channel_2L/max_SIMD))*max(1, int(channel_1L/max_PE)), True),
    (min(channel_1L, max_SIMD), min(channel_1L, max_PE), 9 * max(1, int(channel_1L/max_SIMD))*max(1, int(channel_1L/max_PE)), True),
    
    (min(channel_1L, max_SIMD), min(channel_1L, max_PE), 1 * max(1, int(channel_1L/max_SIMD))*max(1, int(channel_1L/max_PE)), True),
    
    # (min(channel_1L, max_SIMD), 8, 9 * max(1, int(channel_1L/max_SIMD)), True),
    # (8, 8, 9, True),
    
    # (8, 4, 1, True)
    (min(channel_1L, max_SIMD), 8, 9 * max(1, int(channel_1L/max_SIMD)), True),
    (8, 8, 9, True),
    
    (8, 4, 1, True)
] # For the hw, the max PE is set to 32, so the output channels larger than 32 should be set to 32 and the according wmem should be doubled.
print(weights_shape)

weights_shape_dat = []
ws_simd = 64
ws_pe = 32
ws_tile = 2

for index, (simd, pe, wmem, is_padding) in enumerate(weights_shape):
    new_weights_shape = (ws_simd, pe, (wmem + wmem % 2), is_padding)
    weights_shape_dat.append(new_weights_shape)

print(weights_shape_dat)

# weight_file_mode can be selected from {hls_header, decoupled_npy, decoupled_verilog_dat}
# this weight_file_mode only affect the generated weight files instead of FINN synthesis.
weight_file_mode = "decoupled_verilog_dat"
# mem_mode can be selected from {"const", "decoupled", "external"}
# mem_mode will affect the FINN synthesis
mem_mode = "decoupled"

for index, (fcl, (simd, pe, wmem, is_padding), (new_simd, new_pe, new_wmem, new_is_padding)) in enumerate(zip(fc_layers, weights_shape, weights_shape_dat)):
    fcl_inst = MVAU(fcl)
    fcl_inst.set_nodeattr("SIMD", new_simd)
    fcl_inst.set_nodeattr("PE", new_pe)
    fcl_inst.set_nodeattr("mem_mode", mem_mode)
    fcl_inst.set_nodeattr("runtime_writeable_weights", 0)
    fcl_inst.set_nodeattr("ram_style", "block")
    
    weights = model.get_initializer(fcl_inst.onnx_node.input[1])
    weight_tensor = weights.T
    if fcl_inst.get_weight_datatype() == DataType["BIPOLAR"]:
        # convert bipolar to binary
        weight_tensor = (weight_tensor + 1) / 2

    weight_tensor = interleave_matrix_outer_dim_from_partitions(weight_tensor, pe)
    weight_tensor = weight_tensor.reshape(1, pe, wmem, simd) #set pe, wmem, simd accordingly

    if "decoupled" in weight_file_mode:
        new_shape = (1, new_pe, new_wmem, new_simd)
        new_weights = np.zeros(new_shape)
        new_weights[:, :, :weight_tensor.shape[2], :weight_tensor.shape[3]] = weight_tensor
    else:
        new_weights = weight_tensor

    weight_tensor = np.flip(new_weights, axis=-1)
    #collected_tensors.append(weight_tensor)
    export_wdt = DataType[fcl_inst.get_nodeattr("weightDataType")]
    if export_wdt == DataType["BIPOLAR"]:
        export_wdt = DataType["BINARY"]
    if weight_file_mode == "hls_header":
        weight_hls_code = numpy_to_hls_code(weight_tensor, export_wdt, "weights", True, True)
        weight_h_filename = "{}/MVAU_params{}.h".format(save_path, index)
        weight_cpp_filename = "{}/MVAU_params{}.cpp".format(save_path, index)
        f_weights = open(weight_cpp_filename, "w")
        f_weights.write("#include \"MVAU_params{}.h\"\n".format(index))
        f_weights.write("\n")
        if export_wdt.bitwidth() != 1:
            f_weights.write(
                "const FixedPointWeights<{},{},{},{}> mvau_weights{} = \n".format(
                    simd,
                    export_wdt.get_hls_datatype_str(),
                    pe,
                    wmem,
                    index
                )
            )
        else:
            f_weights.write(
                "const BinaryWeights<{},{},{}> mvau_weights{} = \n".format(
                    new_simd,
                    new_pe,
                    new_wmem,
                    index
                )
            )
        #f_weights.write("{")
        f_weights.write(weight_hls_code)
        #f_weights.write("}")
        f_weights.close()

        f_weights = open(weight_h_filename, "w")
        f_weights.write("#ifndef PARAMS{}_H\n".format(index))
        f_weights.write("#define PARAMS{}_H\n".format(index))
        f_weights.write("\n")
        f_weights.write("#include \"../weights.hpp\"\n")
        f_weights.write("\n")
        f_weights.write(
            "extern const FixedPointWeights<{},{},{},{}> mvau_weights{};\n".format(
                simd,
                export_wdt.get_hls_datatype_str(),
                pe,
                wmem,
                index
            )
        )
        f_weights.write("\n")
        f_weights.write("#endif")
        f_weights.close()
    elif "decoupled" in weight_file_mode:
        # create a weight stream for various flavors of decoupled mode:
        # transpose weight tensor from (1, PE, WMEM, SIMD) to (1, WMEM, PE, SIMD)
        # weight_tensor_unflipped = np.transpose(weight_tensor, (0, 2, 1, 3))
        weight_tensor_unflipped = np.transpose(weight_tensor, (0, 1, 2, 3))
        # reverse SIMD flip for saving weights in .npy
        weight_tensor_simd_flipped = np.flip(weight_tensor_unflipped, axis=-1)
        # PE flip for saving weights in .dat
        weight_tensor_unflipped = weight_tensor_unflipped.reshape(1, new_pe, -1, ws_tile, new_simd)
        weight_tensor_pe_flipped = np.flip(weight_tensor_unflipped, axis=-2)
        # reshape weight tensor (simd_flipped and pe_flipped) to desired shape
        # simd_flipped
        # print(index)
        # print(weight_tensor_simd_flipped.shape)
        weight_tensor_simd_flipped = weight_tensor_simd_flipped.reshape(1, -1, ws_tile * new_simd)
        weight_tensor_simd_flipped = weight_tensor_simd_flipped.copy()
        # flipped
        # weight_tensor_pe_flipped = weight_tensor_pe_flipped.reshape(1, -1, ws_tile * new_simd)
        weight_tensor_pe_flipped = weight_tensor_pe_flipped.reshape(1, -1, ws_tile * new_simd)
        weight_tensor_pe_flipped = weight_tensor_pe_flipped.copy()
        weight_file_name_npy = "{}/weights_{}.npy".format(save_path_npy, index)
        weight_file_name_dat = "{}/memstream_{}.dat".format(save_path_dat, index)
        if weight_file_mode == "decoupled_npy":
            # save weight stream into npy for cppsim
            np.save(weight_file_name_npy, weight_tensor_simd_flipped)
        elif weight_file_mode == "decoupled_verilog_dat":
            # convert weight values into hexstring
            # weight_width = fcl_inst.get_weightstream_width()
            weight_stream_width = weight_tensor_pe_flipped.shape[-1] * 4
            # pad to nearest 4 bits to get hex strings
            weight_width_padded = roundup_to_integer_multiple(weight_stream_width, 4)
            weight_tensor_pe_flipped = pack_innermost_dim_as_hex_string(
                weight_tensor_pe_flipped, export_wdt, weight_width_padded, prefix=""
            )
            # add zeroes to pad out file to 1024 entries
            weight_stream = weight_tensor_pe_flipped.flatten()
            weight_stream = weight_stream.copy()
            with open(weight_file_name_dat, "w") as f:
                for val in weight_stream:
                    f.write(val + "\n")
print("Complete.")
model.save(model_dir + "/unet_dataflow_model_01.onnx")

In [ ]:
from qonnx.util.basic import roundup_to_integer_multiple

# Generate MVAU thresh.h to save activation value

# set up loaded model and save_path
model = ModelWrapper(model_dir + "/unet_dataflow_model_01.onnx")
save_path = os.path.join(model_dir, "MVAU_params")
# Check if the directory exists, if not, create it
if not os.path.exists(save_path):
    os.makedirs(save_path)
fc_layers = model.get_nodes_by_op_type("MatrixVectorActivation")

max_SIMD = 64
max_PE = 32
input_channels = 2
channel_1L = 16
channel_2L = 32
channel_3L = 64

thresh_shape = [
    (min(channel_1L, max_PE), channel_1L, 1 * max(1, int(channel_1L/max_PE)), True),
    (min(channel_1L, max_PE), channel_1L, 1 * max(1, int(channel_1L/max_PE)), True),

    (min(channel_2L, max_PE), channel_2L, 1 * max(1, int(channel_2L/max_PE)), True),
    (min(channel_2L, max_PE), channel_2L, 1 * max(1, int(channel_2L/max_PE)), True),

    (min(channel_3L, max_PE), channel_3L, 1 * max(1, int(channel_3L/max_PE)), True),
    (min(channel_3L, max_PE), channel_3L, 1 * max(1, int(channel_3L/max_PE)), True),

    (min(channel_2L, max_PE), channel_2L, 1 * max(1, int(channel_2L/max_PE)), False),

    (min(channel_1L, max_PE), channel_1L, 1 * max(1, int(channel_1L/max_PE)), True),
    (min(channel_1L, max_PE), channel_1L, 1 * max(1, int(channel_1L/max_PE)), True),
    
    (min(channel_1L, max_PE), channel_1L, 1 * max(1, int(channel_1L/max_PE)), False),

    (8, 8, 1, True),
    (8, 8, 1, True),
]
print(thresh_shape)
accDateType_list = []

# select between "hls_header" and "decoupled_verilog_dat"
thresh_file_mode = "decoupled_verilog_dat"

for index, (fcl, (pe, mh, tmem, _)) in enumerate(zip(fc_layers, thresh_shape)):
    fcl_inst = MVAU(fcl)
    fcl_inst.set_nodeattr("PE", pe)
    fcl_inst.minimize_accumulator_width(model)
    accDateType_list.append(fcl_inst.get_nodeattr("accDataType"))
print(accDateType_list)
accDateType_list.pop(-1)
max_bw_str = max(accDateType_list, key=lambda s: int(s[3:]))
max_bw_int = int(max_bw_str[3:])
max_bw_int = roundup_to_integer_multiple(max_bw_int, 4)
max_bw_datatype = DataType[f'INT{max_bw_int}']
tdt_hls_usrset = "ap_int<{}>".format(max_bw_int)
for index, (fcl, (pe, mh, tmem, is_padding)) in enumerate(zip(fc_layers, thresh_shape)):
    fcl_inst = MVAU(fcl)
    fcl_inst.minimize_accumulator_width(model)
    if len(fcl_inst.onnx_node.input) > 2:
        thresholds = model.get_initializer(fcl_inst.onnx_node.input[2])
        if thresholds is not None:
            #fcl_inst.set_nodeattr("PE", pe)
            #fcl_inst.set_nodeattr("MH", mh)
            # tmem = mh // pe
            n_thres_steps = thresholds.shape[1]
            threshold_tensor = thresholds
            #if threshold_tensor.shape[0] == 1:
            #    threshold_tensor = np.tile(threshold_tensor, (mh, 1))
            threshold_tensor = interleave_matrix_outer_dim_from_partitions(threshold_tensor, pe)
            threshold_tensor = threshold_tensor.reshape(1, pe, tmem, n_thres_steps)
            tdt = DataType[fcl_inst.get_nodeattr("accDataType")]
            if thresh_file_mode == "hls_header":
                thresholds_hls_code = numpy_to_hls_code(threshold_tensor, max_bw_datatype, "thresholds", False, True)
                tdt_hls = tdt.get_hls_datatype_str()
                # tdt_hls_usrset = "ap_int<18>"
                # use binary to export bipolar activations
                export_odt = fcl_inst.get_output_datatype()
                odt_hls = export_odt.get_hls_datatype_str()
                
                # write thresholds into thresh.h
                thresh_h_filename = "{}/MVAU_thresh{}.h".format(save_path, index)
                thresh_cpp_filename = "{}/MVAU_thresh{}.cpp".format(save_path, index)
    # cpp file  
                f_thresh = open(thresh_cpp_filename, "w")
                f_thresh.write("#include \"MVAU_thresh{}.h\"\n".format(index))
                f_thresh.write("\n")
                f_thresh.write(
                    "ThresholdsActivation<{},{},{},{},{},{},{}> mvau_threshs{} = \n".format(
                        tmem,
                        pe,
                        threshold_tensor.shape[-1],
                        tdt_hls_usrset,
                        odt_hls,
                        fcl_inst.get_nodeattr("ActVal"),
                        "comp::less_equal<%s, %s>" % (tdt_hls_usrset, tdt_hls_usrset),
                        index,
                    )
                )
                f_thresh.write(thresholds_hls_code)
                f_thresh.close()
    # header file
                f_thresh = open(thresh_h_filename, "w")
                f_thresh.write("#ifndef MVAU_THRESH{}_H\n".format(index))
                f_thresh.write("#define MVAU_THRESH{}_H\n".format(index))
                f_thresh.write("\n")
                f_thresh.write("#include \"../activations.hpp\"\n")
                f_thresh.write("\n")
                f_thresh.write(
                    "extern ThresholdsActivation<{},{},{},{},{},{},{}> mvau_threshs{};\n".format(
                        tmem,
                        pe,
                        threshold_tensor.shape[-1],
                        tdt_hls_usrset,
                        odt_hls,
                        fcl_inst.get_nodeattr("ActVal"),
                        "comp::less_equal<%s, %s>" % (tdt_hls_usrset, tdt_hls_usrset),
                        index,
                    )
                )
                f_thresh.write("\n")
                f_thresh.write("#endif\n")
                f_thresh.close()
            elif "decoupled" in thresh_file_mode:
                threshold_tensor = threshold_tensor.reshape(1, pe, tmem, n_thres_steps)
                if is_padding:
                    new_tmem = 2
                    new_pe = 32
                    new_shape = (1, new_pe, new_tmem, n_thres_steps)
                    new_threshs = np.zeros(new_shape)
                    new_threshs[:, :threshold_tensor.shape[1], :threshold_tensor.shape[2], :] = threshold_tensor
                else:
                    new_pe = 32
                    new_shape = (1, new_pe, tmem, n_thres_steps)
                    new_threshs = np.zeros(new_shape)
                    new_threshs[:, :threshold_tensor.shape[1], :, :] = threshold_tensor

                threshold_tensor = np.transpose(new_threshs, (0, 1, 2, 3))
                threshold_tensor = np.flip(threshold_tensor, axis=-1)
                print(threshold_tensor.shape)
                idimlen = threshold_tensor.shape[-1]
                # idimbits = idimlen * max_bw_datatype.bitwidth()
                idimbits = idimlen * 12
                idimbits = roundup_to_integer_multiple(idimbits, 4)
                # threshold_stream = pack_innermost_dim_as_hex_string(threshold_tensor, max_bw_datatype, idimbits, prefix="")
                threshold_stream = pack_innermost_dim_as_hex_string(threshold_tensor, DataType['INT12'], idimbits, prefix="")
                threshold_stream = threshold_stream.flatten()
                threshold_stream = threshold_stream.copy()

                thresh_file_name_dat = "{}/thresh_memstream_{}.dat".format(save_path, index)
                with open(thresh_file_name_dat, "w") as f:
                    for val in threshold_stream:
                        f.write(val + "\n")
         

model.save(model_dir + "/unet_dataflow_model_02.onnx")

In [ ]:
# Generate Thresholding_Batch thresh.h to save activation value

model = ModelWrapper(model_dir + "/unet_dataflow_model_02.onnx")
save_path = os.path.join(model_dir, "Thresholding_params")
# Check if the directory exists, if not, create it
if not os.path.exists(save_path):
    os.makedirs(save_path)
Thresholding_layers = model.get_nodes_by_op_type("Thresholding_Batch")

# channel_1L = 32
# channel_2L = 64
# channel_3L = 128

thresh_shape = [
    (10, 10, 1),

    (channel_1L, channel_1L, 1),
    (channel_2L, channel_2L, 1),

    (channel_2L, channel_2L, 1),
    (channel_2L, channel_2L, 1),

    (channel_1L, channel_1L, 1),
    (channel_1L, channel_1L, 1),
]

for index, (fcl, (pe, mh, tmem)) in enumerate(zip(Thresholding_layers, thresh_shape)):
    fcl_inst = Thresholding_Batch(fcl)
    fcl_inst.set_nodeattr("PE", pe)
    fcl_inst.set_nodeattr("NumChannels", mh)
    fcl_inst.set_nodeattr("weightDataType", "INT32")
    fcl_inst.set_nodeattr("ram_style", "block")
    thresholds = model.get_initializer(fcl_inst.onnx_node.input[1])
    n_thres_steps = thresholds.shape[1]
    assert n_thres_steps == fcl_inst.get_nodeattr("numSteps"), "Mismatch in threshold steps"
    if not fcl_inst.get_input_datatype().signed():
        # ensure all thresholds are nonnegative
        assert (thresholds >= 0).all()
    # ensure all thresholds are integer
    assert np.equal(np.mod(thresholds, 1), 0).all(), "Need int threshold tensor"
    #fcl_inst.set_nodeattr("PE", pe)
    #fcl_inst.set_nodeattr("MH", mh)
    # tmem = mh // pe
    threshold_tensor = thresholds
    if threshold_tensor.shape[0] == 1:
        threshold_tensor = np.tile(threshold_tensor, (mh, 1))
    threshold_tensor = interleave_matrix_outer_dim_from_partitions(threshold_tensor, pe)
    threshold_tensor = threshold_tensor.reshape(1, pe, tmem, n_thres_steps)
    tdt = fcl_inst.get_weight_datatype()
    print(tdt)
    thresholds_hls_code = numpy_to_hls_code(threshold_tensor, tdt, "thresholds", False, True)
    tdt_hls = tdt.get_hls_datatype_str()
    if tdt_hls == "ap_int<32>":
        tdt_hls = "ap_int<10>"
    # tdt_hls_usrset = "ap_int<8>"
    # use binary to export bipolar activations
    export_odt = fcl_inst.get_output_datatype()
    odt_hls = export_odt.get_hls_datatype_str()
    
    thresh_h_filename = "{}/Thresholding_thresh{}.h".format(save_path, index)
    thresh_cpp_filename = "{}/Thresholding_thresh{}.cpp".format(save_path, index)
# Write to cpp file which contain the data
    f_thresh = open(thresh_cpp_filename, "w")
    f_thresh.write("#include \"Thresholding_thresh{}.h\"\n".format(index))
    f_thresh.write("\n")
    f_thresh.write(
        "ThresholdsActivation<{},{},{},{},{},{},{}> thresB_threshs{} = \n".format(
            tmem,
            pe,
            threshold_tensor.shape[-1],
            tdt_hls,
            odt_hls,
            fcl_inst.get_nodeattr("ActVal"),
            "comp::less_equal<%s, %s>" % (tdt_hls, tdt_hls),
            index,
        )
    )
    f_thresh.write(thresholds_hls_code)
    f_thresh.close()
# Write to header file
    f_thresh = open(thresh_h_filename, "w")
    tdt_hls = tdt.get_hls_datatype_str()
    if tdt_hls == "ap_int<32>":
        tdt_hls = "ap_int<10>"
    # tdt_hls_usrset = "ap_int<8>"
    # use binary to export bipolar activations
    export_odt = fcl_inst.get_output_datatype()
    odt_hls = export_odt.get_hls_datatype_str()
    f_thresh.write("#ifndef THRESHOLDING_THRESH{}_H\n".format(index))
    f_thresh.write("#define THRESHOLDING_THRESH{}_H\n".format(index))
    f_thresh.write("\n")
    f_thresh.write("#include \"../activations.hpp\"\n")
    f_thresh.write("\n")
    f_thresh.write(
        "extern ThresholdsActivation<{},{},{},{},{},{},{}> thresB_threshs{};\n".format(
            tmem,
            pe,
            threshold_tensor.shape[-1],
            tdt_hls,
            odt_hls,
            fcl_inst.get_nodeattr("ActVal"),
            "comp::less_equal<%s, %s>" % (tdt_hls, tdt_hls),
            index,
        )
    )
    f_thresh.write("\n")
    f_thresh.write("#endif\n")
    #f_thresh.write(thresholds_hls_code)
    f_thresh.close()

model.save(model_dir + "/unet_dataflow_model_03.onnx")

In [ ]:
import sys
from ZynqBuild_custom import InsertAndSetFIFODepths_custom

pynq_board = "RFSoC4x2"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10
model_dir = os.environ['FINN_ROOT'] + "/notebooks/HLS_unet/model_onnx_save"
model = ModelWrapper(model_dir + "/unet_dataflow_model_03.onnx")
model = model.transform(InsertAndSetFIFODepths(fpga_part))
model.save(model_dir + "/unet_dataflow_model_03_withfifo.onnx")

model = ModelWrapper(model_dir + "/unet_dataflow_model_03_withfifo.onnx")
model = model.transform(ZynqBuild(platform = pynq_board, period_ns = target_clk_ns))
model.save(model_dir + "/unet_dataflow_model_03_ZynqBuild.onnx")

In [ ]:
import fnmatch

def find_path(base_path, partial_name):
    for root, dirs, files in os.walk(base_path):
        for folder in dirs:
            # Match folder that starts with "code_gen_ipgen_" and ends with the exact partial_name
            if fnmatch.fnmatch(folder, f"code_gen_ipgen_{partial_name}*"):
                return os.path.join(root, folder)
    return None  # If no matching folder is found

In [ ]:
import qonnx.custom_op.registry as registry
from finn.util.fpgadataflow import is_fpgadataflow_node

model = ModelWrapper(model_dir + "/unet_dataflow_model_03.onnx")
base_path = "/tmp/finn_dev_ranhuo"
for node in model.graph.node:
    node_name = node.name
    op_type = node.op_type
    if is_fpgadataflow_node(node):
        inst = registry.getCustomOp(node)
        code_gen_dir_ipgen = find_path(base_path, node_name)
        ipgen_path = code_gen_dir_ipgen + f"/project_{node_name}"
        syn_path = os.path.join(ipgen_path, "sol1", "syn")
        inst.set_nodeattr("code_gen_dir_ipgen", code_gen_dir_ipgen)
        inst.set_nodeattr("ipgen_path", ipgen_path)
        if os.path.isdir(syn_path):
            ip_path = os.path.join(ipgen_path, "sol1", "impl", "ip")
            inst.set_nodeattr("ip_path", ip_path)
        else:
            ip_path = None
model.save(model_dir + "/unet_dataflow_model_03_addpath.onnx")
